# CAPM and Factor Model Compare

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import pandas_datareader.data as web
import statsmodels.api as sm

In [2]:
ticker = ['DEEPAKNTR.NS','^NSEI']
mydata = pd.DataFrame()
for t in ticker:
    mydata [t] = yf.download(t,start='2010-10-1',auto_adjust=False)['Adj Close']


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [3]:
mydata.head()

,DEEPAKNTR.NS,^NSEI
Date,,
2010-10-01,14.873259,6143.399902
2010-10-04,15.023621,6159.450195
2010-10-05,15.073744,6145.799805
2010-10-06,15.303463,6186.450195
2010-10-07,14.998561,6120.299805


In [4]:
mydata.rename(columns={
    'DEEPAKNTR.NS': 'DEEPAKNTR',
    '^NSEI': "NIFTY50"
}, inplace=True)

In [5]:
mydata.head()

,DEEPAKNTR,NIFTY50
Date,,
2010-10-01,14.873259,6143.399902
2010-10-04,15.023621,6159.450195
2010-10-05,15.073744,6145.799805
2010-10-06,15.303463,6186.450195
2010-10-07,14.998561,6120.299805


In [6]:
returns = np.log(mydata/mydata.shift(1))

In [7]:
mkt_return = returns.mean()*250*100
mkt_return ['NIFTY50']

8.57387890531772

# Stock's Beta

In [8]:
cov_ = returns.cov()*250
cov_

,DEEPAKNTR,NIFTY50
DEEPAKNTR,0.146603,0.021926
NIFTY50,0.021926,0.027277


In [9]:
DEEPAKNTR_Beta = (cov_.loc['NIFTY50','DEEPAKNTR']) / (returns['NIFTY50'].var()*250)
DEEPAKNTR_Beta

0.8038296407804323

# CAPM

In [10]:
expt_return = 6.69 + DEEPAKNTR_Beta*(mkt_return ['NIFTY50']-6.69)
CAPM=expt_return

# Fama–French 3 factors

In [11]:
ff_data = pd.read_csv ('C:/users/jubin/onedrive/desktop/F-F_Research_Data_Factors_daily.csv', skiprows=3)

In [12]:
ff_data.tail()

,Unnamed: 0,Mkt-RF,SMB,HML,RF
26125,20251124,1.61,0.30,-0.96,0.02
26126,20251125,1.04,1.65,0.04,0.02
26127,20251126,0.69,-0.06,-0.07,0.02
26128,20251128,0.54,-0.42,0.36,0.02
26129,Copyright 2025 Eugene F. Fama and Kenneth R. F...,NaN,NaN,NaN,NaN


In [13]:
ff_data.rename(columns = {'Unnamed: 0': 'Date'}, inplace = True)  # OR you can rename as ff_data.rename(columns={ff_data.columns[0]: 'Date'}, inplace=True)


In [14]:
ff_data.head()

,Date,Mkt-RF,SMB,HML,RF
0,19260701,0.09,-0.25,-0.27,0.01
1,19260702,0.45,-0.33,-0.06,0.01
2,19260706,0.17,0.30,-0.39,0.01
3,19260707,0.09,-0.58,0.02,0.01
4,19260708,0.22,-0.38,0.19,0.01


In [15]:
ff_data = ff_data[ff_data['Date'].str.isnumeric()]

In [16]:
ff_data.head()

,Date,Mkt-RF,SMB,HML,RF
0,19260701,0.09,-0.25,-0.27,0.01
1,19260702,0.45,-0.33,-0.06,0.01
2,19260706,0.17,0.30,-0.39,0.01
3,19260707,0.09,-0.58,0.02,0.01
4,19260708,0.22,-0.38,0.19,0.01


In [17]:
ff_data.dtypes #date data type is oject right now 

Date       object
Mkt-RF    float64
SMB       float64
HML       float64
RF        float64
dtype: object

In [18]:
ff_data['Date'] = pd.to_datetime(ff_data['Date'], format='%Y%m%d')

In [19]:
ff_data.dtypes #now after when I convert the date in datetime it's data type changed to datetime 

Date      datetime64[ns]
Mkt-RF           float64
SMB              float64
HML              float64
RF               float64
dtype: object

In [20]:
ff_data.set_index('Date', inplace=True)

In [21]:
ff_data.dtypes #now I have set date to index it got remove on below list

Mkt-RF    float64
SMB       float64
HML       float64
RF        float64
dtype: object

In [22]:
ff_data = ff_data/100

In [23]:
data = pd.merge(
    returns[['DEEPAKNTR']],
    ff_data[['Mkt-RF','SMB','HML','RF']],
    left_index=True,
    right_index=True,
    how='inner'
)
data = data.dropna()

In [24]:
data['Excess_Return'] = data['DEEPAKNTR'] - data['RF']
data

,DEEPAKNTR,Mkt-RF,SMB,HML,RF,Excess_Return
Date,,,,,,
2010-10-04,0.010059,-0.0088,-0.0067,-0.0010,0.0000,0.010059
2010-10-05,0.003331,0.0211,0.0077,0.0018,0.0000,0.003331
2010-10-06,0.015125,-0.0011,-0.0035,0.0042,0.0000,0.015125
2010-10-07,-0.020125,-0.0016,0.0003,-0.0026,0.0000,-0.020125
2010-10-08,0.024482,0.0074,0.0094,-0.0015,0.0000,0.024482
...,...,...,...,...,...,...
2025-11-21,-0.007361,0.0103,0.0166,0.0074,0.0002,-0.007561
2025-11-24,-0.040101,0.0161,0.0030,-0.0096,0.0002,-0.040301
2025-11-25,-0.027951,0.0104,0.0165,0.0004,0.0002,-0.028151


In [25]:
X = data[['Mkt-RF', 'SMB', 'HML']]
X = sm.add_constant(X)   # Adds intercept (alpha)

y = data['Excess_Return']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          Excess_Return   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     17.11
Date:                Mon, 08 Jun 2026   Prob (F-statistic):           4.92e-11
Time:                        11:42:00   Log-Likelihood:                 8339.7
No. Observations:                3623   AIC:                        -1.667e+04
Df Residuals:                    3619   BIC:                        -1.665e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0010      0.000      2.563      0.0

In [26]:
E_SMB = data['SMB'].mean()*250
E_HML = data['HML'].mean()*250
E_Mkt_RF = data['Mkt-RF'].mean()*250

In [27]:
betas = model.params
betas

const     0.001033
Mkt-RF    0.245362
SMB      -0.093413
HML       0.163579
dtype: float64

In [28]:
expected_return_DEEPAKNTR = (.0669+ DEEPAKNTR_Beta * E_Mkt_RF + betas['SMB'] * E_SMB
    + betas['HML'] * E_HML)*100

In [29]:
Fama_French_3_factors = expected_return_DEEPAKNTR

In [30]:
print("CAPM:", round(CAPM, 2), "%")
print("Fama–French 3 factors:", round(Fama_French_3_factors, 2), "%")

CAPM: 8.2 %
Fama–French 3 factors: 16.21 %
